# Data Integration Phase
Enriches data from silver lake and writes aggregated and enriched taxi_trips data as gold Delta table `integrated_taxi_trips`.


## 1. Configure Spark


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6f437980-64c6-4e6d-ba37-675bd57dc643;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 115ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs


## 2. Load silver tables

Read silver delta tables.

In [ ]:
from pyspark.sql import functions as F

trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

26/09/09 13:15:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


taxi_trips       18,961,429 rows
weather               7,710 rows
air_quality         164,445 rows
taxi_zones              265 rows


## 3. Hourly weather (NYC local)



In [3]:
hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("temperature_c").alias("temperature_c"),
        F.avg("wind_speed_ms").alias("wind_speed_ms"),
    )
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)


hourly_weather: 5,681 hours
+-----------+-----------+-------------+-------------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms|
+-----------+-----------+-------------+-------------+
|2025-02-16 |5          |1.7          |4.1          |
|2025-02-16 |14         |3.3          |0.75         |
|2025-03-17 |11         |14.15        |0.0          |
+-----------+-----------+-------------+-------------+
only showing top 3 rows



## 4. Hourly air quality (NYC local)



In [ ]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("value").alias("pm25"),
        F.first("unit").alias("pm25_unit"),
    )
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)


hourly_aq: 8,759 hours
+-----------+-----------+------------------+---------------------------+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |
+-----------+-----------+------------------+---------------------------+
|2025-01-01 |0          |13.55             |Micrograms/cubic meter (LC)|
|2025-01-01 |1          |10.7625           |Micrograms/cubic meter (LC)|
|2025-01-01 |2          |11.912499999999998|Micrograms/cubic meter (LC)|
+-----------+-----------+------------------+---------------------------+
only showing top 3 rows



## 5. Pickup / dropoff zone lookups

`taxi_zones` is a small table ==> we can easily and with minimal overhead split them into `pickup_zones` and `dropoff_zones`


In [5]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)


## 6. Enrich trips and write `integrated_taxi_trips`

Left-join weather and air_quality data onto taxi_trip data.

In [6]:
integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .select(
        "taxi_type",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough",
        "dropoff_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "temperature_c",
        "wind_speed_ms",
        "pm25",
        "pm25_unit",
        "pickup_date",
        "pickup_hour",
    )
)

write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
show_delta(spark, GOLD / "integrated_taxi_trips")


integrated_taxi_trips: 18961429 rows @ data/lake/gold/integrated_taxi_trips
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-------------------+--------------+-------------------+-------------------+---------------+-----------+----------+------------+------------+-------------+-------------+------+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone        |pickup_borough|dropoff_location_id|dropoff_zone       |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c|wind_speed_ms|pm25  |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-------------------+--------------+-------------------+-------------------+---------------+-----------+----------+------------+-----------

## 7. Two storage designs

To compare two storage designs, we can partition the same rows aggregated from silver to gold layer in two different ways:

| Partitioning Logic | Table | Partition | Suited for |
| --- | --- | --- | --- |
| By timestamp | `integrated_taxi_trips` | `pickup_date` | average duration per day |
| By location | `integrated_taxi_trips_by_borough` | `pickup_borough` | trip data per borough location |


In [ ]:
import time

gold = read_delta(spark, GOLD / "integrated_taxi_trips")

t0 = time.perf_counter()
write_gold(gold, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")


def storage_report(table_name: str) -> None:
    path = GOLD / table_name
    files = [f for f in path.rglob("*.parquet") if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    n_parts = len({f.parent for f in files})
    print(
        f"{table_name:40} files={len(files):>5}  partitions={n_parts:>4}  size={size_mb:>8.1f} MB"
    )


print()
print("Storage")
storage_report("integrated_taxi_trips")
storage_report("integrated_taxi_trips_by_borough")


write by_borough: 17.6s

Storage
----------------------------------------
integrated_taxi_trips                    files=  443  partitions= 157  size=   465.2 MB
integrated_taxi_trips_by_borough         files=  139  partitions=   8  size=   451.4 MB


### Queries on both designs


In [9]:
import time


def queries(df):
    duration_min = (
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    ) / 60.0
    return {
        "trips per borough": (
            df.groupBy("pickup_borough")
            .agg(F.count(F.lit(1)).alias("trips"))
            .orderBy(F.desc("trips"))
        ),
        "avg duration per day": (
            df.withColumn("duration_min", duration_min)
            .groupBy("pickup_date")
            .agg(F.avg("duration_min").alias("avg_duration_min"))
            .orderBy("pickup_date")
        ),
        "avg fare per borough": (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("avg_fare"))
            .orderBy("pickup_borough")
        ),
    }


def run_queries(table_name: str) -> None:
    spark.catalog.clearCache()
    df = read_delta(spark, GOLD / table_name)
    print(f"\n{table_name}")
    print("-" * 40)
    for name, q in queries(df).items():
        t0 = time.perf_counter()
        rows = q.collect()
        elapsed = time.perf_counter() - t0
        print(f"\n{name}  ({elapsed:.2f}s, {len(rows):,} rows)")
        spark.createDataFrame(rows).show(20, truncate=False)


run_queries("integrated_taxi_trips")
run_queries("integrated_taxi_trips_by_borough")



integrated_taxi_trips
----------------------------------------



trips per borough  (1.01s, 8 rows)


+--------------+--------+
|pickup_borough|trips   |
+--------------+--------+
|Manhattan     |16486609|
|Queens        |1721344 |
|Brooklyn      |570889  |
|Bronx         |133363  |
|Unknown       |38192   |
|N/A           |7454    |
|EWR           |2016    |
|Staten Island |1562    |
+--------------+--------+




avg duration per day  (1.45s, 157 rows)
+-----------+-------------------+
|pickup_date|avg_duration_min   |
+-----------+-------------------+
|2007-12-05 |17.0               |
|2009-01-01 |25.474999999999998 |
|2024-12-25 |0.03333333333333333|
|2024-12-29 |17.45              |
|2024-12-31 |14.76066666666667  |
|2025-01-01 |15.609852961742053 |
|2025-01-02 |16.915689523832487 |
|2025-01-03 |16.065138736299335 |
|2025-01-04 |15.06453791156963  |
|2025-01-05 |14.827447703371156 |
|2025-01-06 |15.308618129029368 |
|2025-01-07 |14.913071977708395 |
|2025-01-08 |14.797202941625418 |
|2025-01-09 |15.496823451085946 |
|2025-01-10 |15.209765819403058 |
|2025-01-11 |13.785585752974244 |
|2025-01-12 |13.926031191008917 |
|2025-01-13 |14.925257468273719 |
|2025-01-14 |15.036757210857543 |
|2025-01-15 |15.370316278918276 |
+-----------+-------------------+
only showing top 20 rows


avg fare per borough  (0.74s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+-----


avg duration per day  (0.85s, 157 rows)
+-----------+-------------------+
|pickup_date|avg_duration_min   |
+-----------+-------------------+
|2007-12-05 |17.0               |
|2009-01-01 |25.474999999999998 |
|2024-12-25 |0.03333333333333333|
|2024-12-29 |17.45              |
|2024-12-31 |14.760666666666669 |
|2025-01-01 |15.609852961742058 |
|2025-01-02 |16.915689523832384 |
|2025-01-03 |16.065138736299307 |
|2025-01-04 |15.06453791156971  |
|2025-01-05 |14.827447703371112 |
|2025-01-06 |15.308618129029233 |
|2025-01-07 |14.913071977708455 |
|2025-01-08 |14.797202941625452 |
|2025-01-09 |15.496823451085987 |
|2025-01-10 |15.209765819403055 |
|2025-01-11 |13.785585752974125 |
|2025-01-12 |13.926031191008882 |
|2025-01-13 |14.925257468273808 |
|2025-01-14 |15.036757210857479 |
|2025-01-15 |15.370316278918175 |
+-----------+-------------------+
only showing top 20 rows


avg fare per borough  (0.48s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+-----